# Phần 3 — Tự hiện thực TransformerEncoder

**Mục tiêu:** Xây dựng lại TransformerEncoder từ các phép toán cơ bản:
- `nn.Linear`, `nn.LayerNorm`, `torch.einsum`
- **Không dùng** `nn.TransformerEncoderLayer` hay `nn.TransformerEncoder`

Sau đó so sánh **Custom ViT** với **PyTorch ViT** từ Phần 1.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from src.data import get_cifar100_loaders, get_device
from src.models_part1 import SimpleViT
from src.models_part3 import CustomMultiHeadAttention, CustomTransformerEncoderLayer, CustomViT
from src.train import fit, load_best_model
from src.utils import (get_param_count, get_predictions, compute_metrics,
                       plot_training_curves, plot_multi_curves, print_results_table,
                       save_metrics_json, load_metrics_json)

DEVICE = get_device()
print(f"Thiết bị: {DEVICE}")
train_loader, val_loader, test_loader, class_names = get_cifar100_loaders(batch_size=128)

## 1. Cơ chế Attention — Giải thích trực quan

**Self-Attention là gì?**

Mỗi token (patch) hỏi: *"Tôi nên chú ý đến những token nào khác?"*

**3 vector cho mỗi token:**
- **Query (Q):** "Tôi đang tìm kiếm gì?"
- **Key (K):**   "Tôi có thể cung cấp gì?"
- **Value (V):** "Thông tin thực sự của tôi là gì?"

**Công thức:**
```
scores = Q @ K^T / sqrt(d_head)   # Độ tương đồng giữa mọi cặp tokens
attn   = softmax(scores, dim=-1)  # Normalize thành phân phối xác suất
output = attn @ V                  # Tổng có trọng số của Values
```

**Multi-Head:** Thực hiện Attention song song với `num_heads` bộ Q,K,V khác nhau
→ Mỗi head học 1 loại quan hệ khác nhau

In [ ]:
# Minh hoạ Attention trên toy example (T=5 tokens)
torch.manual_seed(42)
d_model, num_heads, T = 8, 2, 5
d_head = d_model // num_heads

# Tạo input ngẫu nhiên
x_toy = torch.randn(1, T, d_model)  # [1, 5, 8]

# Custom MHA
mha = CustomMultiHeadAttention(d_model, num_heads)
with torch.no_grad():
    # Lấy attention weights để visualize
    Q = mha.W_q(x_toy).reshape(1, T, num_heads, d_head).transpose(1, 2)
    K = mha.W_k(x_toy).reshape(1, T, num_heads, d_head).transpose(1, 2)
    scores = torch.einsum('bhid,bhjd->bhij', Q, K) / (d_head ** 0.5)
    attn_weights = F.softmax(scores, dim=-1)

fig, axes = plt.subplots(1, num_heads, figsize=(10, 4))
for h in range(num_heads):
    im = axes[h].imshow(attn_weights[0, h].numpy(), cmap='Blues', vmin=0, vmax=1)
    axes[h].set_title(f'Head {h+1}\nAttention Matrix')
    axes[h].set_xlabel('Key (token j)')
    axes[h].set_ylabel('Query (token i)')
    plt.colorbar(im, ax=axes[h])

plt.suptitle('Attention weights: token i "chú ý" đến token j bao nhiêu?', fontsize=11)
plt.tight_layout()
plt.show()
print(f"Kiểm tra: mỗi hàng tổng = 1.0?  {attn_weights.sum(dim=-1).mean():.6f}")

## 2. Hiện thực Custom Multi-Head Attention

Giải thích chi tiết về einsum notation:

```python
# 'bhid,bhjd->bhij' nghĩa là:
# b = batch, h = head, i = query position, j = key position, d = d_head
# Với mỗi (b, h, i): tính dot product với mọi j
# → scores[b, h, i, j] = sum_d(Q[b,h,i,d] * K[b,h,j,d])

scores = torch.einsum('bhid,bhjd->bhij', Q, K) / sqrt(d_head)

# 'bhij,bhjd->bhid' nghĩa là:
# Với mỗi (b, h, i): tổng có trọng số của Values
# → out[b, h, i, d] = sum_j(attn[b,h,i,j] * V[b,h,j,d])
out = torch.einsum('bhij,bhjd->bhid', attn, V)
```

In [ ]:
# Kiểm tra Custom MHA vs nn.MultiheadAttention
# (Không thể so sánh output trực tiếp vì khởi tạo weights khác nhau,
#  nhưng có thể kiểm tra: shape giống nhau + attention distribution hợp lệ)

custom_mha = CustomMultiHeadAttention(d_model=64, num_heads=4)
pytorch_mha = nn.MultiheadAttention(embed_dim=64, num_heads=4, batch_first=True)

x_test = torch.randn(2, 10, 64)  # [batch=2, seq=10, d_model=64]

with torch.no_grad():
    custom_out = custom_mha(x_test)
    pytorch_out, attn_weights = pytorch_mha(x_test, x_test, x_test)

print(f"Custom MHA output shape:  {custom_out.shape}")
print(f"PyTorch MHA output shape: {pytorch_out.shape}")
print(f"Shapes khớp nhau: {custom_out.shape == pytorch_out.shape}")
print()

# Kiểm tra attention là phân phối hợp lệ (mỗi hàng sum=1)
# Tính lại attention weights từ custom MHA để verify
with torch.no_grad():
    Q = custom_mha.W_q(x_test).reshape(2, 10, 4, 16).transpose(1, 2)
    K = custom_mha.W_k(x_test).reshape(2, 10, 4, 16).transpose(1, 2)
    scores = torch.einsum('bhid,bhjd->bhij', Q, K) / (16 ** 0.5)
    attn = F.softmax(scores, dim=-1)

print(f"Attention row sums (should all be ~1.0):")
print(f"  Mean: {attn.sum(dim=-1).mean():.6f}")
print(f"  Min:  {attn.sum(dim=-1).min():.6f}")
print(f"  Max:  {attn.sum(dim=-1).max():.6f}")
print("✓ Attention distribution hợp lệ!")

## 3. Xây dựng Custom ViT

In [ ]:
model_custom_vit = CustomViT(num_classes=100)
print(model_custom_vit)
print(f"\nSố tham số Custom ViT: {get_param_count(model_custom_vit)}")

# So sánh với PyTorch ViT
model_pytorch_vit = SimpleViT(num_classes=100)
print(f"Số tham số PyTorch ViT: {get_param_count(model_pytorch_vit)}")
print(f"\n(Hai mô hình có cùng hyperparams: d_model=128, heads=4, layers=4, patch_size=4)")

## 4. Huấn luyện và So sánh

In [ ]:
TRAIN_MODE = True

histories_p3 = {}

# ── PyTorch ViT (load từ Part 2 nếu đã train) ──
ckpt_vit = '../exercise/results/checkpoints/vit.pt'
if os.path.exists('../exercise/results/metrics/vit_history.json'):
    histories_p3["SimpleViT (PyTorch)"] = load_metrics_json('../exercise/results/metrics/vit_history.json')
    print("Loaded PyTorch ViT history từ Part 2")
else:
    model_vit = SimpleViT(100).to(DEVICE)
    histories_p3["SimpleViT (PyTorch)"] = fit(
        model_vit, train_loader, val_loader,
        {"epochs": 100, "lr": 3e-4, "device": DEVICE, "save_path": ckpt_vit}
    )

# ── Custom ViT ──
ckpt_custom = '../exercise/results/checkpoints/custom_vit.pt'
config_custom = {"epochs": 100, "lr": 3e-4, "device": DEVICE, "save_path": ckpt_custom}

if TRAIN_MODE:
    print("=" * 50)
    print("Training: CustomViT")
    print("=" * 50)
    model_custom_vit = CustomViT(100).to(DEVICE)
    histories_p3["CustomViT (Tự xây)"] = fit(model_custom_vit, train_loader, val_loader, config_custom)
    save_metrics_json(histories_p3["CustomViT (Tự xây)"],
                      '../exercise/results/metrics/custom_vit_history.json')
else:
    histories_p3["CustomViT (Tự xây)"] = load_metrics_json(
        '../exercise/results/metrics/custom_vit_history.json')

print("✓ Done")

In [ ]:
# Vẽ training curves
plot_multi_curves(
    list(histories_p3.values()),
    list(histories_p3.keys()),
    title="So sánh: SimpleViT (PyTorch) vs CustomViT (Tự xây)",
    save_path='../exercise/results/plots/part3_comparison_curves.png'
)

In [ ]:
# Bảng kết quả
results_p3 = {}
vit_pairs = {
    "SimpleViT (PyTorch)": (SimpleViT(100), ckpt_vit),
    "CustomViT (Tự xây)": (CustomViT(100), ckpt_custom),
}

for name, (m, ckpt) in vit_pairs.items():
    if os.path.exists(ckpt):
        m = load_best_model(m, ckpt, DEVICE)
        preds, labels = get_predictions(m, test_loader, DEVICE)
        metrics = compute_metrics(preds, labels)
        best_val = max(histories_p3[name]["val_acc"])
        results_p3[name] = {
            "test_acc": metrics["accuracy"],
            "val_acc": best_val,
            "f1_macro": metrics["f1_macro"],
            "params": get_param_count(m),
        }

save_metrics_json(results_p3, '../exercise/results/metrics/part3_results.json')
print_results_table(results_p3)

## 5. Nhận xét

**Kết quả dự kiến:**
- Hai mô hình có accuracy rất gần nhau (sai lệch < 1-2%)
- Điều này **xác nhận** hiện thực Custom Transformer là đúng!

**Tại sao không giống hệt nhau?**
- Khởi tạo weights ngẫu nhiên khác nhau → convergence khác nhau
- `nn.TransformerEncoderLayer` dùng `F.scaled_dot_product_attention` (tối ưu hơn)
- Thứ tự áp dụng LayerNorm có thể khác nhau đôi chút

**Bài học:**
- Hiện thực từ đầu giúp hiểu sâu cơ chế bên trong
- Kết quả tương đương xác nhận tính đúng đắn
- Framework có sẵn (PyTorch) được tối ưu hóa tốt hơn về tốc độ